In [45]:
import pandas as pd
import json

In [46]:
df_terms_final = pd.read_parquet('cleaned_aws_terms.parquet')
df_products = pd.read_parquet('cleaned_aws_products.parquet')

In [47]:
print("Terms shape:", df_terms_final.shape)
print("Products shape:", df_products.shape)
df_products.head(3)

Terms shape: (99979, 5)
Products shape: (100000, 44)


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.currentGeneration,attributes.instanceFamily,attributes.vcpu,attributes.physicalProcessor,...,attributes.volumeName,attributes.deploymentModel,attributes.engineMajorVersion,attributes.extendedSupportPricingYear,attributes.processorFeatures,attributes.storageMedia,attributes.minVolumeSize,attributes.maxVolumeSize,attributes.limitlesspreview,attributes.acu
0,SJCPEREKG4AJC3P7,Database Instance,AmazonRDS,Canada (Central),AWS Region,db.r8i.xlarge,Yes,Memory optimized,4,Intel Xeon Scalable (Granite Rapids),...,None,None,None,None,None,None,None,None,None,None
1,JYX35YJ8Z5DXXS98,Database Instance,AmazonRDS,Asia Pacific (Mumbai),AWS Region,db.r5d.24xlarge,Yes,Memory optimized,96,Intel Xeon Platinum 8175,...,None,None,None,None,None,None,None,None,None,None
2,H4WDB4TZHXB8URCU,Database Instance,AmazonRDS,Asia Pacific (Melbourne),AWS Region,db.t3.medium,Yes,General purpose,2,Intel Skylake E5 2686 v5 (2.5 GHz),...,None,None,None,None,None,None,None,None,None,None


In [48]:
if 'attributes.servicename' in df_products.columns:
    df_products.drop(columns=['attributes.servicename'], errors='ignore', inplace=True)

print (f"Dimensions (rows, cols): {df_products.shape}")

Dimensions (rows, cols): (100000, 43)


In [49]:
df_database_instance= df_products[df_products['productFamily'] == 'Database Instance'].reset_index(drop=True)


#### Ανάλυση Database Instance

In [50]:
id_columns_db_instance = ['sku', 'productFamily', 
    'attributes.servicecode',
    'attributes.location', 
    'attributes.locationType']

In [51]:
feature_columns_db_instance = ['attributes.instanceType', 'attributes.instanceFamily', 
                                'attributes.vcpu', 'attributes.memory', 
                                'attributes.storage', 'attributes.physicalProcessor', 
                                'attributes.networkPerformance', 'attributes.databaseEngine', 
                                'attributes.databaseEdition', 'attributes.licenseModel', 
                                'attributes.deploymentOption']

In [52]:
total_db_instance = []
total_db_instance.extend(id_columns_db_instance)
total_db_instance.extend(feature_columns_db_instance)
df_database_instance[total_db_instance].head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.instanceFamily,attributes.vcpu,attributes.memory,attributes.storage,attributes.physicalProcessor,attributes.networkPerformance,attributes.databaseEngine,attributes.databaseEdition,attributes.licenseModel,attributes.deploymentOption
0,SJCPEREKG4AJC3P7,Database Instance,AmazonRDS,Canada (Central),AWS Region,db.r8i.xlarge,Memory optimized,4,32 GiB,EBS Only,Intel Xeon Scalable (Granite Rapids),Up to 12500 Megabit,SQL Server,Standard,Bring your own media,Multi-AZ
1,JYX35YJ8Z5DXXS98,Database Instance,AmazonRDS,Asia Pacific (Mumbai),AWS Region,db.r5d.24xlarge,Memory optimized,96,768 GiB,4 x 900 NVMe SSD,Intel Xeon Platinum 8175,25 Gbps,MySQL,None,No license required,Single-AZ
2,H4WDB4TZHXB8URCU,Database Instance,AmazonRDS,Asia Pacific (Melbourne),AWS Region,db.t3.medium,General purpose,2,4 GiB,EBS Only,Intel Skylake E5 2686 v5 (2.5 GHz),Low to Moderate,MySQL,None,No license required,Multi-AZ
3,R8Q2435NXQCAMNX9,Database Instance,AmazonRDS,US East (Ohio),AWS Region,db.x1e.8xlarge,Memory optimized,32,976 GiB,1 x 960 SSD,Intel Xeon E7-8880 v3,Up to 10 Gigabit,Oracle,Enterprise,Bring your own license,Multi-AZ
4,U86UU5KVR2Z9BH2T,Database Instance,AmazonRDS,Africa (Cape Town),AWS Region,db.r5.4xlarge,Memory optimized,16,128 GiB,Aurora IO Optimization Mode,Intel Xeon Platinum 8175,Up to 10 Gigabit,Aurora PostgreSQL,None,No license required,Single-AZ


In [ ]:
remaining_cols_rds = [col for col in df_database_instance.columns if col not in total_db_instance]

df_final_database_instance = df_database_instance[total_db_instance].copy()

df_final_database_instance['additional_attributes'] = df_database_instance[remaining_cols_rds].apply(
    lambda row: json.dumps({k: v for k, v in row.to_dict().items() if pd.notna(v)}), 
    axis=1
)

In [57]:
df_master_database_instance = pd.merge(
    df_final_database_instance, 
    df_terms_final, 
    on='sku', 
    how='inner'
)

df_master_database_instance = df_master_database_instance.reset_index(drop=True)

In [60]:
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)
df_master_database_instance.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.instanceFamily,attributes.vcpu,attributes.memory,attributes.storage,attributes.physicalProcessor,attributes.networkPerformance,attributes.databaseEngine,attributes.databaseEdition,attributes.licenseModel,attributes.deploymentOption,additional_attributes,rateCode,description,unit,priceUSD
0,SJCPEREKG4AJC3P7,Database Instance,AmazonRDS,Canada (Central),AWS Region,db.r8i.xlarge,Memory optimized,4,32 GiB,EBS Only,Intel Xeon Scalable (Granite Rapids),Up to 12500 Megabit,SQL Server,Standard,Bring your own media,Multi-AZ,"{""attributes.currentGeneration"": ""Yes"", ""attri...",SJCPEREKG4AJC3P7.JRTCKXETXF.6YS6EN2CT7,USD 1.08 per db.r8i.xlarge Multi-AZ instance h...,Hrs,1.0800
1,JYX35YJ8Z5DXXS98,Database Instance,AmazonRDS,Asia Pacific (Mumbai),AWS Region,db.r5d.24xlarge,Memory optimized,96,768 GiB,4 x 900 NVMe SSD,Intel Xeon Platinum 8175,25 Gbps,MySQL,None,No license required,Single-AZ,"{""attributes.currentGeneration"": ""Yes"", ""attri...",JYX35YJ8Z5DXXS98.JRTCKXETXF.6YS6EN2CT7,$ 15.89 per RDS db.r5d.24xlarge Single-AZ inst...,Hrs,15.8900
2,H4WDB4TZHXB8URCU,Database Instance,AmazonRDS,Asia Pacific (Melbourne),AWS Region,db.t3.medium,General purpose,2,4 GiB,EBS Only,Intel Skylake E5 2686 v5 (2.5 GHz),Low to Moderate,MySQL,None,No license required,Multi-AZ,"{""attributes.currentGeneration"": ""Yes"", ""attri...",H4WDB4TZHXB8URCU.JRTCKXETXF.6YS6EN2CT7,USD 0.224 db.t3.medium Multi-AZ instance hour ...,Hrs,0.2240
3,R8Q2435NXQCAMNX9,Database Instance,AmazonRDS,US East (Ohio),AWS Region,db.x1e.8xlarge,Memory optimized,32,976 GiB,1 x 960 SSD,Intel Xeon E7-8880 v3,Up to 10 Gigabit,Oracle,Enterprise,Bring your own license,Multi-AZ,"{""attributes.currentGeneration"": ""Yes"", ""attri...",R8Q2435NXQCAMNX9.JRTCKXETXF.6YS6EN2CT7,USD 22.418 per RDS db.x1e.8xlarge Multi-AZ ins...,Hrs,22.4179
4,U86UU5KVR2Z9BH2T,Database Instance,AmazonRDS,Africa (Cape Town),AWS Region,db.r5.4xlarge,Memory optimized,16,128 GiB,Aurora IO Optimization Mode,Intel Xeon Platinum 8175,Up to 10 Gigabit,Aurora PostgreSQL,None,No license required,Single-AZ,"{""attributes.currentGeneration"": ""Yes"", ""attri...",U86UU5KVR2Z9BH2T.JRTCKXETXF.6YS6EN2CT7,USD 3.962 per RDS db.r5.4xlarge IO-optimized S...,Hrs,3.9620


#### Ανάλυση

In [53]:
df_products['productFamily'].unique()

array(['Database Instance', 'ServerlessV2', 'Performance Insights',
       'Provisioned IOPS', None, 'Database Storage',
       'Provisioned Throughput', 'RDSProxy', 'CPU Credits',
       'Storage Snapshot', 'System Operation', 'Limitless', 'Serverless',
       'Aurora Global Database'], dtype=object)

In [ ]:
summary_data = []
for col in df_database_instance.columns:
    sample_vals = list(df_database_instance[col].dropna().unique()[:5])
    summary_data.append({'Column': col, 'Sample_Values': sample_vals})

df_summary = pd.DataFrame(summary_data)

# Εμφάνιση χωρίς περικοπές
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
df_summary

,Column,Sample_Values
0,sku,"[SJCPEREKG4AJC3P7, JYX35YJ8Z5DXXS98, H4WDB4TZHXB8URCU, R8Q2435NXQCAMNX9, U86UU5KVR2Z9BH2T]"
1,productFamily,[Database Instance]
2,attributes.servicecode,[AmazonRDS]
3,attributes.location,"[Canada (Central), Asia Pacific (Mumbai), Asia Pacific (Melbourne), US East (Ohio), Africa (Cape Town)]"
4,attributes.locationType,"[AWS Region, AWS Outposts]"
5,attributes.instanceType,"[db.r8i.xlarge, db.r5d.24xlarge, db.t3.medium, db.x1e.8xlarge, db.r5.4xlarge]"
6,attributes.currentGeneration,"[Yes, No]"
7,attributes.instanceFamily,"[Memory optimized, General purpose, Compute optimized, Micro instances]"
8,attributes.vcpu,"[4, 96, 2, 32, 16]"
9,attributes.physicalProcessor,"[Intel Xeon Scalable (Granite Rapids), Intel Xeon Platinum 8175, Intel Skylake E5 2686 v5 (2.5 GHz), Intel Xeon E7-8880 v3, Intel Xeon Scalable (Sapphire Rapids 8488C)]"
